# My PDF AI · Motor Vetorial (Caminho B)

**Pixel-perfect.** Esse notebook usa PyMuPDF pra editar PDFs no nível vetorial — remove objetos antigos do stream e injeta os novos sem tocar no fundo, watermarks ou linhas de tabela.

## Como usar
1. Rode todas as células em ordem (Runtime → Run all)
2. Suba o **PDF 1** (template/base) quando pedido
3. Suba o **PDF 2** (origem dos dados novos) quando pedido
4. **Marque os checkboxes** dos campos que quer substituir
5. **Revise** os valores extraídos no formulário antes de gerar
6. Baixe o PDF resultante

## 1 · Setup

In [ ]:
# Instala dependências (Colab geralmente já tem ipywidgets)
!pip install -q pymupdf ipywidgets

# Cria estrutura my_pdf_ai/ no Colab (mesma estrutura do projeto real)
import os, shutil
BASE = 'my_pdf_ai'
if os.path.exists(BASE): shutil.rmtree(BASE)
os.makedirs(f'{BASE}/core', exist_ok=True)
os.makedirs(f'{BASE}/api',  exist_ok=True)
open(f'{BASE}/__init__.py','w').close()
open(f'{BASE}/core/__init__.py','w').close()
open(f'{BASE}/api/__init__.py','w').close()
print('OK · estrutura criada')

## 2 · Cole os módulos do `core/`

Os arquivos `.py` do projeto vão ficar inline aqui pra o notebook ser **self-contained**.
Em produção (Caminho C) eles vivem no repositório — você só importa.

**Cole abaixo o conteúdo de cada arquivo enviado junto com este notebook:**
- `core/models.py` → célula 2.1
- `core/barcode_gen.py` → célula 2.2
- `core/detector.py` → célula 2.3
- `core/extractor.py` → célula 2.4
- `core/editor.py` → célula 2.5

(O notebook vem com os arquivos separados. Se você preferir tudo num lugar, em vez de colar, rode as células de import abaixo.)

In [ ]:
# 2.bis · Modo automático: lê os .py do diretório atual e instala em my_pdf_ai/core
# Se você subir os arquivos models.py, barcode_gen.py, detector.py, extractor.py, editor.py
# pra mesma pasta do notebook, essa célula copia tudo pra estrutura certa.
import os, shutil, glob
for src in ['models.py','barcode_gen.py','detector.py','extractor.py','editor.py']:
    if os.path.exists(src):
        shutil.copy(src, f'my_pdf_ai/core/{src}')
        print(f'✓ copiado {src}')
    else:
        print(f'⚠ {src} não encontrado — cole o conteúdo manualmente nas células 2.x')

## 3 · Upload do PDF Template (PDF 1)

In [ ]:
from google.colab import files
print('Sube o PDF 1 (template/base)...')
uploaded = files.upload()
template_filename = list(uploaded.keys())[0]
template_bytes = uploaded[template_filename]
print(f'\n✓ Template carregado: {template_filename}  ·  {len(template_bytes):,} bytes')

## 4 · Detecção automática dos campos no template

In [ ]:
from my_pdf_ai.core.detector import detect_all
from my_pdf_ai.core.models import FieldKey

detections = detect_all(template_bytes)
print(f'Campos detectados no template:\n')
for key in FieldKey:
    campos = detections.get(key, [])
    if campos:
        print(f'  ✓ {key.value:20s}  {len(campos)} ocorrência(s)  texto: {campos[0].raw_text!r}')
        for i, c in enumerate(campos):
            print(f'      #{i}  bbox=({c.bbox.x0:.0f},{c.bbox.y0:.0f})–({c.bbox.x1:.0f},{c.bbox.y1:.0f})')
    else:
        print(f'  ✗ {key.value:20s}  (não detectado)')

## 5 · Upload do PDF de Dados (PDF 2)

In [ ]:
print('Suba o PDF 2 (origem dos dados novos)...')
uploaded = files.upload()
source_filename = list(uploaded.keys())[0]
source_bytes = uploaded[source_filename]
print(f'\n✓ Source carregado: {source_filename}  ·  {len(source_bytes):,} bytes')

## 6 · Extração automática + revisão

In [ ]:
from my_pdf_ai.core.extractor import extract_from_pdf

extracted = extract_from_pdf(source_bytes)
print('Extraído do PDF 2:\n')
print(f'  Valor              : {extracted.valor or "(não encontrado)"}')
print(f'  Vencimento         : {extracted.vencimento or "(não encontrado)"}')
print(f'  Pagador            : {extracted.pagador or "(não encontrado)"}')
print(f'  Linha digitável    : {extracted.linha_digitavel or "(não encontrada)"}')
print(f'  Código de barras   : {extracted.codigo_barras or "(não derivado)"}')
print(f'\n⤓ Revise abaixo no formulário (célula 7) antes de gerar.')

## 7 · Formulário de revisão + checkboxes

Marque APENAS os campos que quer substituir. Ajuste os valores se a extração automática errou.

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML

display(HTML('<h3 style="margin:0">Campos editáveis</h3>'
             '<p style="color:#666;margin:4px 0 12px 0">Marque o que quer substituir e revise os valores.</p>'))

def make_row(label, value, key):
    cb = W.Checkbox(value=bool(value), description='', indent=False, layout=W.Layout(width='28px'))
    txt = W.Text(value=value or '', layout=W.Layout(width='420px'), placeholder=f'(novo valor para {label})')
    lbl = W.HTML(f'<b style="width:160px;display:inline-block">{label}</b>')
    box = W.HBox([cb, lbl, txt])
    return box, cb, txt, key

rows = [
    make_row('Valor',           extracted.valor,           FieldKey.VALOR),
    make_row('Vencimento',      extracted.vencimento,      FieldKey.VENCIMENTO),
    make_row('Linha digitável', extracted.linha_digitavel, FieldKey.LINHA_DIGITAVEL),
    make_row('Código de barras (44 dígitos)', extracted.codigo_barras, FieldKey.CODIGO_BARRAS),
]
for box, _, _, _ in rows: display(box)
print()  # respiro visual

## 8 · Aplicar edições e baixar o PDF resultante

In [ ]:
from my_pdf_ai.core.editor import apply_edits
from my_pdf_ai.core.models import BoletoData, EditRequest

# Lê estado dos checkboxes/textos
data = BoletoData()
fields_to_edit = []
for _, cb, txt, key in rows:
    if cb.value and txt.value.strip():
        setattr(data, key.value, txt.value.strip())
        fields_to_edit.append(key)

if not fields_to_edit:
    print('⚠ Nenhum campo selecionado. Marque pelo menos uma checkbox e rode esta célula de novo.')
else:
    print(f'Editando {len(fields_to_edit)} campo(s): {[k.value for k in fields_to_edit]}\n')
    req = EditRequest(template_pdf_bytes=template_bytes, data=data, fields_to_edit=fields_to_edit)
    report = apply_edits(req)

    counts = getattr(report, 'edited_count', {})
    print('  Editados:')
    for k in report.edited:
        n = counts.get(k, 1)
        print(f'    · {k.value}: {n} ocorrência(s) substituída(s)')
    if report.skipped:
        print(f'  Skipped:')
        for k, reason in report.skipped.items():
            print(f'    · {k.value}: {reason}')

    out_name = template_filename.replace('.pdf', '_editado.pdf')
    with open(out_name, 'wb') as f:
        f.write(report.output_pdf_bytes)
    print(f'\n✓ Arquivo gerado: {out_name}  ·  {len(report.output_pdf_bytes):,} bytes')
    files.download(out_name)